<a href="https://colab.research.google.com/github/Mariano-rr/ThinkPythonAssignments/blob/main/Week15.5/NutritionToolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas
import pandas as pd
from datetime import datetime
import os

class NutritionTracker:
    def __init__(self, filename="macro_log.csv"):
        self.filename = filename
        self.goals = {'p': 120, 'c': 200, 'f': 70}  # Daily gram goals
        self._init_file()

    def _init_file(self):
        """Ensures CSV exists with proper headers for efficient reading."""
        if not os.path.exists(self.filename):
            df = pd.DataFrame(columns=['date', 'food', 'p', 'c', 'f', 'calories'])
            df.to_csv(self.filename, index=False)

    def log_meal(self, name, p, c, f):
        """Vectorized logic to calculate calories and append data."""
        # Standard conversion: 4 cal/g for P/C, 9 cal/g for F
        calories = (p * 4) + (c * 4) + (f * 9)
        new_row = pd.DataFrame([[datetime.now().date(), name, p, c, f, calories]],
                               columns=['date', 'food', 'p', 'c', 'f', 'calories'])

        # 'a' mode (append) is most efficient for large logs
        new_row.to_csv(self.filename, mode='a', header=False, index=False)
        print(f"\n[✓] Saved: {name} ({calories} kcal)")

    def view_progress(self):
        """Uses Pandas filtering to instantly aggregate today's totals."""
        df = pd.read_csv(self.filename)
        if df.empty:
            print("\nNo entries found.")
            return

        today = str(datetime.now().date())
        # Efficiency: Filter entire dataset at once rather than looping
        todays_meals = df[df['date'] == today]

        if todays_meals.empty:
            print("\nNo meals logged today yet!")
            return

        totals = todays_meals[['p', 'c', 'f', 'calories']].sum()

        print(f"\n--- Progress for {today} ---")
        for key in ['p', 'c', 'f']:
            current = totals[key]
            goal = self.goals[key]
            percent = (current / goal) * 100
            print(f"{key.upper()}: {current:>5.1f}g / {goal:>3}g ({percent:>5.1f}%)")
        print(f"CALORIES: {totals['calories']:.0f} kcal")
        print("-" * 30 + "\n")

def main():
    tracker = NutritionTracker()
    print("🚀 Efficient Macro-Nutrient Tracker Active")

    while True:
        choice = input("1: Log Meal | 2: Progress | 3: Exit -> ")
        if choice == "1":
            try:
                name = input("Food: ")
                p, c, f = float(input("P (g): ")), float(input("C (g): ")), float(input("F (g): "))
                tracker.log_meal(name, p, c, f)
            except ValueError:
                print("Error: Please enter numeric values for macros.")
        elif choice == "2":
            tracker.view_progress()
        elif choice == "3":
            break

if __name__ == "__main__":
    main()